# Loss functions

> A set of custom loss functions

In [1]:
#| default_exp losses

In [2]:
#| hide
from nbdev.showdoc import *

# Loss Functions

In [3]:
#| export
from bioMONAI.core import store_attr

In [4]:
#| export
import numpy as np

from monai.losses import SSIMLoss
import torch.nn as nn
from torch import abs, sqrt, div, sigmoid, complex64, where, isinf, zeros_like, real, isnan
from torch.fft import fftshift
from torch.fft import fft2

from bioMONAI.core import torch_from_numpy

In [5]:
show_doc(SSIMLoss)

---

### SSIMLoss

>      SSIMLoss (spatial_dims:int, data_range:float=1.0,
>                kernel_type:monai.metrics.regression.KernelType|str=gaussian,
>                win_size:int|collections.abc.Sequence[int]=11,
>                kernel_sigma:float|collections.abc.Sequence[float]=1.5,
>                k1:float=0.01, k2:float=0.03,
>                reduction:monai.utils.enums.LossReduction|str=mean)

Compute the loss function based on the Structural Similarity Index Measure (SSIM) Metric.

For more info, visit
    https://vicuesoft.com/glossary/term/ssim-ms-ssim/

SSIM reference paper:
    Wang, Zhou, et al. "Image quality assessment: from error visibility to structural
    similarity." IEEE transactions on image processing 13.4 (2004): 600-612.

### Combined Loss

In [6]:
#| export

class CombinedLoss:
    "losses combined"
    def __init__(self, spatial_dims=2, alpha=0.33, beta=0.33):
        store_attr()
        self.SSIM_loss = SSIMLoss(spatial_dims=spatial_dims)
        self.MSE_loss =  nn.MSELoss()
        self.MAE_loss =  nn.L1Loss()
        
    def __call__(self, pred, targ):
        return (1 - self.alpha - self.beta) * self.SSIM_loss(pred, targ) + self.alpha * self.MSE_loss(pred, targ) + self.beta * self.MAE_loss(pred, targ)
        

### Dice Loss

In [7]:
#| export

class DiceLoss(nn.Module):

    """
    DiceLoss computes the Sørensen–Dice coefficient loss, which is often used 
    for evaluating the performance of image segmentation algorithms.

    The Dice coefficient is a measure of overlap between two samples. It ranges 
    from 0 (no overlap) to 1 (perfect overlap). The Dice loss is computed as 
    1 - Dice coefficient, so it ranges from 1 (no overlap) to 0 (perfect overlap).

    Attributes:
        smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.

    Methods:
        forward(inputs, targets):
            Computes the Dice loss between the predicted probabilities (inputs) 
            and the ground truth (targets).
    """

    def __init__(self, smooth=1):

        """
        Initializes the DiceLoss instance with a smoothing factor.

        Args:
            smooth (float): A smoothing factor to avoid division by zero and ensure numerical stability.
                            Default is 1.
        """
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        
        # Make sure the inputs are probabilities
        inputs = sigmoid(inputs)

        # Flatten tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calculate the intersection
        intersection = (inputs * targets).sum()

        # Compute Dice Coefficient
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        # Copmute dice loss
        loss = 1 - dice

        return loss
        

In [8]:
# inputs and targets must be equally dimensional tensors
from torch import randn, randint

inputs = randn((1, 1, 256, 256))  # Input
targets = randint(0, 2, (1, 1, 256, 256)).float()  # Ground Truth

# Initialize
dice_loss = DiceLoss()

# Compute loss
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())



Dice Loss: 0.4994280934333801


### Fourier Ring Correlation

#### Radial mask

In [9]:
#| export

def radial_mask(r, # Radius of the radial mask
                cx=128, # X coordinate mask center
                cy=128, # Y coordinate maske center
                sx=np.arange(0, 256), 
                sy=np.arange(0, 256), 
                delta=1,
               ):

    """
    Generate a radial mask.

    #### Parameters:
       - r (int or float): Radius of the circular mask.
       - cx (int, optional): X-coordinate of the center of the circular mask. Defaults to 128.
       - cy (int, optional): Y-coordinate of the center of the circular mask. Defaults to 128.
       - sx (numpy.ndarray, optional): Array of x-coordinates forming a grid. Defaults to np.arange(0, 256).
       - sy (numpy.ndarray, optional): Array of y-coordinates forming a grid. Defaults to np.arange(0, 256).
       - delta (int or float, optional): Thickness adjustment for the circular mask. Defaults to 1.

    #### Returns:
       - numpy.ndarray: Radial mask.
    """

    # Calculate squared distances from each point in the grid to the center
    ind = (sx[np.newaxis, :] - cx) ** 2 + (sy[:, np.newaxis] - cy) ** 2

    # Define inner boundary of the circular mask
    ind1 = ind <= ((r[0] + delta) ** 2)

    # Define outer boundary of the circular mask
    ind2 = ind > (r[0] ** 2)

    # Create the radial mask by combining inner and outer boundaries
    return ind1 * ind2


In [10]:
#| export

def get_radial_masks(width, height):

    """
    Generates a set of radial masks and corresponding to spatial frequencies.

    #### Parameters:
        - width (int): Width of the image.
        - height (int): Height of the image.

    #### Returns:
        tuple: A tuple containing:
            - numpy.ndarray: Array of radial masks.
            - numpy.ndarray: Array of spatial frequencies corresponding to the masks.
    """

    # Calculate Nyquist frequency
    freq_nyq = int(np.floor(int(min(width, height)) / 2.0))
   
    # Generate radii from 0 to Nyquist frequency
    radii = np.arange(freq_nyq).reshape(freq_nyq, 1)

    # Generate radial masks using the radial_mask function    
    radial_masks = np.apply_along_axis(radial_mask, 1, radii, width/2, height/2, np.arange(0, width), np.arange(0, height), 1)

    # Calculate spatial frequencies
    spatial_freq = radii.astype(np.float32) / freq_nyq
    spatial_freq = spatial_freq / max(spatial_freq)
    spatial_freq = spatial_freq.squeeze(1)

    return radial_masks, spatial_freq


#### Fourier ring correlation

In [11]:
#| export

def get_fourier_ring_correlations(image1, image2):

    
    """
    Compute Fourier Ring Correlation (FRC) between two images.

    #### Args:
        - image1 (torch.Tensor): First input image.
        - image2 (torch.Tensor): Second input image.

    #### Returns:
        tuple: A tuple containing:
            - torch.Tensor: Fourier Ring Correlation values.
            - torch.Tensor: Array of spatial frequencies.
    """
    

    # Get image height and width
    height = image1.shape[len(image1.shape)-1]
    width = image1.shape[len(image1.shape)-2]
    
    # Get set of radial masks, spatial frequency, and Nyquist frequency
    radial_masks, spatial_frequency = get_radial_masks(height,width)

    # Get Nyquist frequency
    freq_nyq = len(spatial_frequency)
    
    # Transform tensor to complex
    image1 = image1.to(complex64)
    image2 = image2.to(complex64)

    # Transofrm array dimensions to (freq_nyq, width. height)
    image1 = image1.unsqueeze(0).repeat(freq_nyq, 1, 1)
    image2 = image2.unsqueeze(0).repeat(freq_nyq, 1, 1)

    # Convert spatial frequency and radial masks to torch.tensor
    spatial_frequency = torch_from_numpy(spatial_frequency)
    radial_masks = torch_from_numpy(radial_masks)

    # Transform tensor to complex
    radial_masks = radial_masks.to(complex64)
         
    # Compute fourier transform
    fft_image1 = fftshift(fft2(image1))
    fft_image2 = fftshift(fft2(image2))

    # Get elements only in the ring
    t1 = fft_image1 * radial_masks
    t2 = fft_image2 * radial_masks
        
    # image2 to complex conjugate
    t2_conj = t2.conj()

    # Numerator
    numerador = abs(real((t1 * t2_conj).sum(dim=(1,2))))

    # Denominator    
    denominador_1 = ((abs(t1) * abs(t1)).sum(dim=(1,2)))
    denominador_2 = ((abs(t2) * abs(t2)).sum(dim=(1,2)))     
    denominador = sqrt(denominador_1 * denominador_2)
   
    # Fourier shell correlation
    FRC = div(numerador, denominador)

    # Remove possible inf and NaN.
    FRC = where(isinf(FRC), zeros_like(FRC), FRC)  # inf
    FRC = where(isnan(FRC), zeros_like(FRC), FRC)  # NaN

    return FRC , spatial_frequency

In [12]:
#| export

def FRCLoss(image1, image2):

    """
    Compute the Fourier Ring Correlation (FRC) loss between two images.

    #### Args:
        - image1 (torch.Tensor): The first input image.
        - image2 (torch.Tensor): The second input image.

    #### Returns:
        - torch.Tensor: The FRC loss.
    """
    
    return (1 - FRCM(image1, image2))
    

In [13]:
#| export

from scipy.optimize import curve_fit


def seventh_fourier_ring_correlation(image1,image2):


    """
    Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

    #### Args:
        - image1 (torch.Tensor): The first input image.
        - image2 (torch.Tensor): The second input image.

    #### Returns:
        - float: The cutoff frequency.
    """

    # Get y and x coordinates
    y, x = get_fourier_ring_correlations(image1, image2)

    # x -> frequency   y -> correlation
    x = x.numpy()
    y = y.numpy()


    # Exponential function to fit
    def exponential_func(x, a, b, c):
        return a * np.exp(-b * x) + c

    # Make fit
    params, params_covariance = curve_fit(exponential_func, x, y, p0=[1, 1, 1])

    # Get Cutoff requency at 1/7
    cutoff_frequency = (exponential_func((1/7), *params))

    return cutoff_frequency

In [14]:
show_doc(seventh_fourier_ring_correlation)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/losses.py#L241){target="_blank" style="float:right; font-size:smaller"}

### seventh_fourier_ring_correlation

>      seventh_fourier_ring_correlation (image1, image2)

Calculate the cutoff frequency at when Fourier ring correlation drops to 1/7.

#### Args:
    - image1 (torch.Tensor): The first input image.
    - image2 (torch.Tensor): The second input image.

#### Returns:
    - float: The cutoff frequency.

---

In [15]:
#| hide
import nbdev; nbdev.nbdev_export()